# 긴축기라는 구분 대신 실제 기준금리를 쓰면 어떨까?

같은 긴축기라도 2022년과 2023년의 금리 수준은 달랐다. 연평균 기준금리를 사용해, 금리가 높은 해에 사전 단기차입이 많은 업종과 적은 업종의 차이가 더 벌어졌는지 살펴본다.

ICR(이자보상배율)은 영업이익을 이자비용으로 나눈 값이다. 3배라면 영업이익이 그해 이자비용의 세 배라는 뜻이다. 한 행은 한 업종의 한 해다. 


[전체 질문 목록](../README.md)

**표 읽는 법**　N은 업종·연도 관측 수, G는 업종 수다. 회귀표의 beta는 추정한 관계의 크기, lo와 hi는 95% 신뢰구간의 양 끝이다. p값은 계수가 0이라는 가정에서 지금만큼 또는 더 극단적인 검정통계량이 나올 확률이다.

# 실제 기준금리 수준을 이용한 대안 분석
2026-11-02 사용자 실행 승인. 기존 analysis_v1 및 원자료는 보존한다.

표본: 제조업 40업종, 2018–2023. 종속변수: ICR(배). 금리: 변경일 당일부터 새 금리를 적용한 일별 기준금리의 연평균(%). 기존 Post/2021 상호작용 대신 사전 E×금리, 사전 L×금리로 연결한다. E,L은 분석 시작 이전인 2015–2017년의 연도별 비율 평균. 3년 평균 관례를 유지하기 위한 분석자 선택이며 유일한 공식 기준이 아니다.

변경의 영향을 구분하기 위해 A=기존 M2(2018–20 평균), B=같은 기간모형+2015–17 평균, C=금리×E만, D=금리×E 및 금리×L을 순서대로 보고한다. D가 이번 대안의 중심이며 기존 M2를 대체한 확정 주 모형은 아니다. 전체 업종 경기 자료는 아직 미확보로 추가하지 않는다. 금리 변화율·시차·장단기 분리 모형은 이번 실행에 포함하지 않는다.

모형 D: Y_it=업종FE+연도FE+β(E_i×r_t)+γ(L_i×r_t)+ε_it. 금리 단독항은 연도FE에 흡수. E,L 단독항은 업종FE에 흡수. β는 E 1%p 차이에 따른 금리 1%p당 상대 ICR 차이(배). 기간모형 계수와 단위가 다르다. 금리 자체의 인과효과를 식별하지 않는다.

업종 군집 CR1+t39 CI; WCR-S/C 9,999회, seed=20261102. 모든 p값은 양측·다중비교 미조정. 240행이지만 금리 시점은 6개이며 업종 간 오차 독립 가정은 남는다. 기존 결과를 확인한 이후의 탐색적 대안 분석이다.

재실행: 이 노트북 폴더에서 Python3.12 및 requirements.txt 환경으로 전체 셀 실행. inputs 원본 복사본의 SHA256을 검증한다.

In [1]:
# 재무 패널 키 확인
from pathlib import Path
import json,hashlib
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from IPython.display import display
from wildboottest.wildboottest import wildboottest
BASE = Path.cwd()
(BASE / 'tables').mkdir(exist_ok=True)
for x in json.loads((BASE/'inputs/manifest.json').read_text(encoding='utf-8')):
 assert hashlib.sha256((BASE/'inputs'/x['file']).read_bytes()).hexdigest()==x['sha256']
def save(d, n):
    d.to_csv(BASE / 'tables' / f'{n}.csv', index=False, encoding='utf-8-sig')
d = pd.read_csv(BASE / 'inputs/panel.csv')
s = pd.read_csv(BASE / 'inputs/stability.csv')
assert len(d)==240 and d.code.nunique()==40 and set(d.year)==set(range(2018,2024))
assert not d.duplicated(['code','year']).any()

# 사전 노출 평균 계산
s['Enew'] = 100 * (s['단기차입'] + s['단기사채']) / s['자산총계']
s['Lnew'] = 100 * s['유동자산'] / s['유동부채']
pre = s[s['연도'].between(2015, 2017)]
assert pre.groupby('업종코드').size().eq(3).all()
av = pre.groupby('업종코드')[['Enew', 'Lnew']].mean()
d['Enew'] = d.code.map(av.Enew)
d['Lnew'] = d.code.map(av.Lnew)

# 일별 금리를 연평균으로 집계
rr = pd.read_csv(BASE / 'inputs/rates.csv')
rr['date'] = pd.to_datetime(rr['변경일자'])
rr = rr.sort_values('date')
assert not rr.date.duplicated().any()
# 이 분석에 필요한 2018–23만 산출. 2024년 이후 기록은 사용하지 않는다.
calendar=pd.DataFrame({'date':pd.date_range('2018-01-01','2023-12-31')})
daily = pd.merge_asof(calendar, rr[['date', '기준금리']], on='date', direction='backward')
assert daily['기준금리'].notna().all()
annual=daily.groupby(daily.date.dt.year)['기준금리'].mean().rename('rate').rename_axis('year').reset_index()
d['rate']=d.year.map(annual.set_index('year').rate)
for v in ['E', 'L']:
    d[v + 'P_new'] = d[v + 'new'] * d.Post
    d[v + 'T_new'] = d[v + 'new'] * (d.year == 2021)
    d[v + 'R'] = d[v + 'new'] * d.rate
assert np.isfinite(d[['Y','Enew','Lnew','rate','ER','LR']]).all().all()
save(annual, 'annual_rates')
save(d, 'analysis_panel')
display(annual)
print('업종별 사전 E/L:',len(av),'금리 연도:',len(annual))


   year      rate
0  2018  1.521918
1  2019  1.582877
2  2020  0.704918
3  2021  0.613014
4  2022  2.032192
5  2023  3.491781
업종별 사전 E/L: 40 금리 연도: 6


### 출력에서 읽을 수 있는 것

2022년 2.032와 2023년 3.492는 그해의 일수 가중 연평균 기준금리(%)다. 각 연도의 마지막 기준금리를 뜻하지 않는다.

- 2018~2023년 240행에서 업종·연도가 중복되지 않는지 확인했다.
- 금리 모형의 사전 재무구조는 2015~2017년 세 해로 고정했다. 기본 모형과 비교할 때 기간 구분 외에 노출도 기준도 달라진다는 점을 반영한다.
- 날짜별 기준금리로 계산한 연평균은 2021년 0.613%, 2022년 2.032%, 2023년 3.492%였다. 이 연간 값과 업종별 사전 노출을 곱해 금리 모형에 넣는다.

In [2]:
# 공통 추정 함수 정의
def design(d,terms):
    X=pd.concat([pd.DataFrame({'const':np.ones(len(d))},index=d.index),d[terms],pd.get_dummies(d.code,prefix='i',drop_first=True,dtype=float),pd.get_dummies(d.year,prefix='y',drop_first=True,dtype=float)],axis=1).astype(float)
    assert np.linalg.matrix_rank(X)==X.shape[1]
    return X
fits = {}
rows = []
def fit(label, d, terms, y='Y'):
    X = design(d, terms)
    model = sm.OLS(d[y].astype(float), X)
    res = model.fit(cov_type='cluster', cov_kwds={'groups': d.code, 'use_correction': True, 'df_correction': True}, use_t=True)
    G = d.code.nunique()
    q = stats.t.ppf(0.975, G - 1)
    for term in terms:
        b = float(res.params[term])
        se = float(res.bse[term])
        rows.append(dict(model=label, outcome=y, term=term, beta=b, se=se, lo=b - q * se, hi=b + q * se, p=float(2 * stats.t.sf(abs(b / se), G - 1)), N=len(d), G=G))
    fits[label] = (d.copy(), terms, model, res)
    return res

# 원래 기간 모형 재현
fit('A_original_post',d,['EP','LP','ET','LT'])

# 사전기간만 교체
fit('B_post_pre1517',d,['EP_new','LP_new','ET_new','LT_new'])

# 금리와 단기차입 결합
fit('C_rate_E',d,['ER'])

# 유동성과 금리 결합 추가
fit('D_rate_EL',d,['ER','LR'])
results = pd.DataFrame(rows)
save(results, 'coefficients')
display(results)
np.testing.assert_allclose(fits['A_original_post'][3].params.EP,0.5459119157113599,atol=1e-9)
# 기존 모형 재현 및 금리 단독항의 흡수 확인
X = design(d, ['ER', 'LR'])
assert np.linalg.matrix_rank(np.column_stack([X, d.rate])) == X.shape[1]
print('기존 M2 reproduction and rate absorption passed')


              model outcome    term      beta  ...        hi         p    N   G
0   A_original_post       Y      EP  0.545912  ...  1.459896  0.234272  240  40
1   A_original_post       Y      LP  0.068685  ...  0.203388  0.308719  240  40
2   A_original_post       Y      ET -0.434758  ...  0.220019  0.187029  240  40
3   A_original_post       Y      LT  0.143002  ...  0.371875  0.213805  240  40
4    B_post_pre1517       Y  EP_new  0.364575  ...  1.147895  0.352293  240  40
5    B_post_pre1517       Y  LP_new  0.067761  ...  0.175710  0.211728  240  40
6    B_post_pre1517       Y  ET_new -0.600370  ...  0.262749  0.167362  240  40
7    B_post_pre1517       Y  LT_new  0.126763  ...  0.320436  0.193248  240  40
8          C_rate_E       Y      ER  0.432142  ...  0.843973  0.040204  240  40
9         D_rate_EL       Y      ER  0.327311  ...  0.699104  0.082753  240  40
10        D_rate_EL       Y      LR -0.027096  ...  0.021843  0.269607  240  40

[11 rows x 10 columns]
既存 M2 reproducti

### 출력에서 읽을 수 있는 것

ER은 사전 단기차입 의존도와 금리의 곱이다. C에서 +0.432배(95% 구간 +0.020~+0.844)였지만 D에서는 +0.327배(−0.045~+0.699)였다.

- 동일한 업종·연도 고정효과를 유지한 채 설명변수만 교체할 수 있게 했다.
- 기존 기간 모형 A의 단기차입 계수는 +0.546배였다.
- 사전기간만 바꾼 B에서는 +0.365배(p=0.352)였다.
- C에서는 연평균 금리가 1%포인트 높을 때 사전 단기차입 의존도가 1%포인트 높은 업종의 상대 ICR이 +0.432배였다(p=0.040). 예상했던 악화와 반대 방향이다.
- 유동성과 금리의 결합항도 넣은 D에서는 +0.327배(p=0.083)로 추정됐다. 공통 기준금리 자체의 효과는 연도 효과와 분리되지 않았다.

## 군집 부트스트랩 및 수치 검산
군집 고정효과만 제거하고 연도 효과는 유지한다. MacKinnon·Nielsen·Webb(2023) §§5–6에 따른 WCR-S(type31)를 우선, WCR-C(type11)를 대조로 보고한다. CI는 CR1이며 bootstrap CI가 아니다. 금리 인상과 인하에서 동일한 선형 기울기를 가정하는 모형이라는 한계가 있다.

In [3]:
# 업종 단위 재검정
boot=[]
for label, terms in [('C_rate_E', ['ER']), ('D_rate_EL', ['ER', 'LR'])]:
    dd, _, model, res = fits[label]
    xx = pd.DataFrame(model.exog, columns=model.exog_names)
    yy = pd.Series(model.endog)
    gg = pd.Series(dd.code.to_numpy())
    xx = xx - xx.groupby(gg).transform('mean')
    yy = yy - yy.groupby(gg).transform('mean')
    xx = xx.loc[:, np.max(abs(xx), axis=0) > 1e-10]
    assert np.linalg.matrix_rank(xx) == xx.shape[1]
    mm = sm.OLS(yy, xx)
    for term in terms:
        np.testing.assert_allclose(mm.fit().params[term], res.params[term], atol=1e-09)
        for variant in ['31', '11']:
            w = wildboottest(mm, param=term, cluster=pd.factorize(gg)[0], B=9999, seed=20261102, bootstrap_type=variant, impose_null=True, weights_type='rademacher', show=False)
            boot.append({'model': label, 'term': term, 'variant': variant, 'p_boot': float(w['p-value'].iloc[0]), 'B': 9999, 'seed': 20261102})
boot = pd.DataFrame(boot)
save(boot, 'bootstrap')
display(boot)

# 행렬 계산으로 검산
# OLS/CR1 행렬 공식 대조
X = fits['D_rate_EL'][2].exog
y = d.Y.to_numpy()
b = np.linalg.lstsq(X, y, rcond=None)[0]
u = y - X @ b
bread = np.linalg.inv(X.T @ X)
sc = np.array([X[d.code == g].T @ u[d.code == g] for g in sorted(d.code.unique())])
cov = bread @ sc.T @ sc @ bread * 40 / 39 * (len(d) - 1) / (len(d) - X.shape[1])
np.testing.assert_allclose(b, fits['D_rate_EL'][3].params, atol=1e-08)
np.testing.assert_allclose(cov, fits['D_rate_EL'][3].cov_params(), atol=1e-07)
print('OLS/CR1 독립 검산 완료')


       model term variant    p_boot     B      seed
0   C_rate_E   ER      31  0.035404  9999  20261102
1   C_rate_E   ER      11  0.035404  9999  20261102
2  D_rate_EL   ER      31  0.060006  9999  20261102
3  D_rate_EL   ER      11  0.061806  9999  20261102
4  D_rate_EL   LR      31  0.237324  9999  20261102
5  D_rate_EL   LR      11  0.235124  9999  20261102
OLS/CR1 독립 검산 완료


### 출력에서 읽을 수 있는 것

variant 31과 11은 두 검정 방식이다. 두 행을 서로 독립적인 연구 결과처럼 세지 않는다.

- C의 부트스트랩 p값은 0.035, D는 0.060이었다. 유동성을 함께 고려하면 양의 관계에 대한 증거가 약해졌다.
- 행렬식으로 다시 구한 D의 계수와 표준오차 계산이 기존 결과와 일치했다.

## 근거·해석 범위
금리 이력: [한국은행 공식 기준금리 추이](https://www.bok.or.kr/portal/singl/baseRate/list.do?menuNo=200643). 입력은 기존 프로젝트의 변경이력 CSV이며 날짜별 적용일수로 연평균을 재계산했다.

금리 수준·전달 시차의 참고 근거: [Kitsul·Lang·Samadi(2023), Introduction 및 §II](https://www.federalreserve.gov/econres/notes/feds-notes/monetary-policy-tightening-and-debt-servicing-costs-of-nonfinancial-companies-20231201.html). 관련 본문 인용: “increased at a much slower rate”. 기존 부채 비용과 신규 차입금리 반응이 다름을 설명한다. 우리 연간 금리 상호작용 모형의 타당성이나 인과성을 직접 입증한 연구는 아니다. 전체를 읽었다는 주장이 아니다.

[MacKinnon 외(2023), §§5–6](https://onlinelibrary.wiley.com/doi/full/10.1002/jae.2969): 군집 고정효과 제거 후 WCR-S 및 WCR-C 사용. 기존 analysis_v1 보고서의 본문 검증·인용 설명 참조.

E/L의 2015–17 평균은 시작 전 재무특성이라는 장점과 2022–23까지 오래된 정보라는 한계가 함께 있다. 기준금리 경로에 선형 기울기를 부여하며 코로나·업종별 수요 등과 상호작용한 누락요인을 배제하지 못한다. 단기차입의 정확한 항목구성·산업 ICR 집계 공식 정의 미확인은 유지된다. 금리변화나 경기 통제를 추가한 모형을 실행한 것으로 해석하지 않는다.

## 실행 결과와 해석 (2026-11-02)

40업종 × 2018–2023년 = 240개 관측치를 사용했다. 기존 자료·노트북은 수정하지 않았다. 기존 M2 계수 재현, 금리 단독항의 연도효과 흡수, OLS 및 군집 공분산 행렬의 독립 계산을 확인했다. 모든 코드 셀을 동일 Python 실행 공간에서 순서대로 실행하고 출력을 저장했다.

| 분석 | 단기차입×금리 계수 | 95% 신뢰구간 | 군집 t p값 | WCR-S p값 |
|---|---:|---|---:|---:|
| C: 단기차입만 | +0.4321 | [0.0203, 0.8440] | 0.0402 | 0.0354 |
| D: 단기차입+유동비율 | +0.3273 | [-0.0445, 0.6991] | 0.0828 | 0.0600 |

D의 유동비율×금리 계수는 -0.0271, 95% CI [-0.0760, 0.0218], 군집 t p=0.2696, WCR-S p=0.2373이다. 계수는 사전 재무비율 1%p 차이와 기준금리 1%p 차이가 결합할 때의 상대 ICR 차이(배)다.

C의 양의 계수는 관례적인 5% 기준에서 유의하지만, 유동비율을 포함한 D에서는 같은 기준을 충족하지 않는다. 5%는 보편적인 진실 판정선이 아니라 관례이며 p값을 그대로 보고한다. 10%로 기준을 바꾸어 가설 채택을 주장하지 않는다. 두 모형 모두 예상했던 단기차입 취약성의 음의 방향과 반대다. 단기차입이 보호 효과를 낸다는 인과적 결론도 낼 수 없다. 유동성의 완충 효과 자체는 이번 모형의 LR항만으로 검정되지 않는다. 이를 보려면 E×L×금리 같은 별도 모형이 필요하며 이번에는 실행하지 않았다.

기존 M2의 EP=0.5459(p=0.2343)를 재현했다. 사전 재무구조만 2015–17년으로 바꾼 기간모형 B의 EP=0.3646(p=0.3523)이다. B와 D 비교는 같은 사전 재무구조에서 기간구분과 금리 경로 표현의 차이를 보여준다. 기간모형과 금리모형은 계수 단위와 시간에 대한 가정이 다르므로 숫자의 크기를 그대로 비교할 수 없다.

### 기존 설정에서 달라진 내용
1. Post 및 2021년 전환 상호작용 → 연평균 실제 기준금리와의 상호작용. 경계 지정 대신 금리의 수준 차이를 쓰되, 인상·인하·코로나 시기에 같은 선형 관계를 가정한다. 연도 고정효과는 유지하지만 2021년의 재무구조별 특수 반응을 별도로 허용하지 않는다.
2. 사전 재무구조 2018–20 평균 → 2015–17 평균. 2018년부터 사용하는 금리 경로보다 먼저 정한 재무구조가 필요하기 때문이다. 3년 평균은 분석자 선택이며, 뒤 시점에서 오래된 정보가 된다는 한계가 있다. 변경 영향을 분리한 B도 함께 보고했다.
3. 기존 분석은 유지하고 이번 결과는 탐색적 대안으로 추가했다. 원자재·환율·생산·출하 지수 및 결합항은 이번 C/D에 넣지 않았다. M4/M5를 다시 돌린 결과가 아니다.

### 다음 분석에 대한 제안 (미반영)
이번 결과만으로 기간구분이 비유의 결과의 원인이었다고 결론낼 수 없다. 이미 요청한 전 업종 생산·출하 지수를 확보한 뒤 경기 차이를 함께 고려할 때 계수가 유지되는지 확인하는 것이 다음 순서다. 2018년 증가율 계산을 위해 2017–2023년 연간 원지수와 업종 분류·기준년·단위·출처 메타정보가 필요하다. 생산과 출하는 각각 넣어 비교하고, 실제 생산은 금리 영향을 받는 경로일 수도 있으므로 인과효과를 정화한 통제로 표현하지 않는다. 승인된 기존 기간모형에 추가하는 작업과 이번 금리모형까지 확장하는 작업은 구분한다.

금리 시점은 6개뿐이다. 업종 40개가 서로 다른 금리 경로를 경험한 것은 아니다. 업종 군집 추론은 업종 간 남은 오차의 독립성을 전제로 하므로, 이번 p값을 확정적 증거로 쓰지 않는다. 기준금리와 실제 차입금리는 다르며 그 전달 정도·시간차를 이 분석에서 직접 측정한 것은 아니다.

### 출처 확인과 적용 범위
- 한국은행 「Monetary Policy Report」 2019년 2월 Executive Summary iii쪽, 6항의 관련 본문을 확인했다. 짧은 인용: “maintained at an annual 1.50% since November of 2017”. 2018년 11월 인상 전 1.50% 유지였음을 확인하는 용도다. [당시 2018-11-30 공식 결정문](https://www.bok.or.kr/eng/bbs/E0000634/view.do?menuNo=400069&nttId=10048600)도 1.50→1.75%를 명시한다. 현재 공식 추이 웹페이지에서 수집된 2018-04-20 2.0% 행은 이 당시 보고서와 상충한다. 해당 행을 입력에 추가하지 않고 당시 자료와 맞는 기존 CSV를 유지했다. 이 불일치를 전체 이력이 모두 새로 검증됐다는 주장으로 대체하지 않는다.
- [Kitsul·Lang·Samadi(2023), FEDS Notes](https://www.federalreserve.gov/econres/notes/feds-notes/monetary-policy-tightening-and-debt-servicing-costs-of-nonfinancial-companies-20231201.html), I–II절 관련 본문: “increased at a much slower rate”. 기존 부채 비용이 신규 차입금리보다 느리게 올랐다는 설명이다. 기준금리와 기업의 실제 부담금리를 구분하는 근거로 사용한다. 해외 상장기업 자료이며 우리의 한국 업종별 ICR 회귀식이나 연평균 선택을 직접 검증한 근거는 아니다. 본문 IV절의 현금/자산은 우리의 유동비율과 다르고, V절의 EBITDA 기반 ICR도 우리의 영업이익 기반 ICR과 다르다.
- 군집 추론은 [MacKinnon·Nielsen·Webb(2023)](https://onlinelibrary.wiley.com/doi/full/10.1002/jae.2969)의 관련 §§5–6을 앞선 검토에서 확인한 방법에 따른다. 이번 연평균 금리·2015–17년 평균·선형 상호작용 선택은 연구팀 검토를 위한 분석 사양이지 해당 논문이 정한 공식 기준이 아니다.
